## Init

In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit
from zoneinfo import ZoneInfo
from pyspark.sql import functions as F

In [0]:
import sys
import os

current_directory = os.getcwd()
if current_directory not in sys.path:
    sys.path.append(current_directory)
    
from script.utils.functions import get_date_from_timestamp, get_time_from_timestamp, get_hour_from_timestamp

## Leer la tabla de ciudades 

In [0]:
df_cities = (
    spark.read
    .table("weather.silver_cities")
    .select("state_code", "lat", "lon")
    .orderBy("state_code")
    # .limit(10)
)

# Convertir las filas a diccionarios 
cities_config = [row.asDict() for row in df_cities.collect()]

## Read from Open-Meteo API 

In [0]:
import time
import requests
from datetime import datetime, date

raw_records = []

def extract_open_meteo_with_retry(lat, lon, date_str, max_retries=3):
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}"
        f"&hourly=temperature_2m,relative_humidity_2m,precipitation"
        f"&start_date={date_str}&end_date={date_str}"
        f"&timezone=auto"
    )
    
    for attempt in range(1, max_retries + 1):
        response = requests.get(url)
        data = response.json()
        times = data.get("hourly", {}).get("time", [])

        if len(times) == 24:
            print(f" Intento {attempt}: Ingesta completa (24/24 horas).")
            return data, "SUCCESS"

        print(f" Intento {attempt}: Ingesta incompleta ({len(times)}/24 horas). Reintentando...")
        time.sleep(5)

    print("Alerta: Se alcanzaron los reintentos máximos. Guardando payload incompleto...")
    return data, "PARTIAL_SUCCESS"

# Iterate over the cities and query Open-Meteo
for c in cities_config:
    data, status = extract_open_meteo_with_retry(c["lat"], c["lon"], date.today().isoformat())

    raw_records.append({
        "state_code": c["state_code"],
        # "request_date": request_date,
        "raw_payload": json.dumps(data),
        "ingested_at": datetime.now().isoformat(),
        "source_endpoint": url
    })

# Convert to DataFrame from PySpark
df = spark.createDataFrame(raw_records)

In [0]:
# Ordernar columnas
df_bronze = (
    df
    .select(
        col("state_code").cast("int"),
        col("raw_payload"),
        col("ingested_at").cast("timestamp"),
        col("source_endpoint")
    )
)

In [0]:
df.display()

## Write data with MERGE using (state_code + date of ingested_at) as unique key 

In [0]:
from delta.tables import DeltaTable

table_name = "weather.raw_data"

if not spark.catalog.tableExists(table_name):
    (
        df_bronze
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
else: 
    target_table = DeltaTable.forName(spark, table_name)

    (
        target_table.alias("target")
        .merge(
            df_bronze.alias("source"), 
            condition="""
            target.state_code = source.state_code AND 
            to_date(target.ingested_at) = to_date(source.ingested_at)
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

## sanity check

In [0]:
%sql
select * 
from weather.raw_data
order by state_code